# Vistas minables

En este notebook armamos las tablas que va a usar cada modelo, a partir de los archivos limpios que
guardamos en `02_limpieza_datos_ml.ipynb`.

Las dos se guardan en `data/vistas/`.

## 1. Preparación del entorno

Para correr este notebook tienen que estar los archivos de `data/clean/`
Usamos `config.py` para las rutas y la conexión a DuckDB.

In [1]:
from config import conectar, ruta_sql

con = conectar()

# Ejecuta una consulta SQL y devuelve el resultado como DataFrame.
def q(consulta):
    return con.sql(consulta).df()

## 2. Datos de partida

Creamos vistas sobre los archivos limpios y contamos cuántos usuarios distintos escuchan a cada
artista.

In [2]:
con.execute(f"CREATE OR REPLACE VIEW events_clean AS SELECT * FROM {ruta_sql('events_clean')}")
con.execute(f"CREATE OR REPLACE VIEW acoustic_clean AS SELECT * FROM {ruta_sql('acoustic_clean')}")
con.execute(f"CREATE OR REPLACE VIEW users_clean AS SELECT * FROM {ruta_sql('users_clean')}")

# Cantidad de usuarios distintos que escucharon a cada artista
con.execute("""
    CREATE OR REPLACE TEMP TABLE oyentes_artista AS
    SELECT artist_id, COUNT(DISTINCT user_id) AS oyentes
    FROM events_clean
    GROUP BY artist_id
""")

q("SELECT * FROM oyentes_artista ORDER BY oyentes DESC LIMIT 10")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,artist_id,oyentes
0,320,19886
1,1602,19140
2,46,17926
3,3,17115
4,4115,17009
5,295,16695
6,27,16338
7,185,16191
8,924,15989
9,470,15548


Antes de unir las tablas revisamos que no haya ids repetidos en usuarios ni en canciones. Si una
canción apareciera dos veces, el `JOIN` duplicaría sus escuchas.

In [3]:
q("""
    SELECT
        (SELECT COUNT(*) - COUNT(DISTINCT user_id)  FROM users_clean)    AS usuarios_repetidos,
        (SELECT COUNT(*) - COUNT(DISTINCT track_id) FROM acoustic_clean) AS canciones_repetidas
""")

,usuarios_repetidos,canciones_repetidas
0,0,0


## 3. Vista para clustering: `vista_usuarios`

Queremos agrupar usuarios según cómo escuchan música, así que cada fila tiene que ser un usuario. Nos quedamos con los que tienen al menos 100 escuchas con datos de audio, para que el
perfil no dependa de pocas canciones.

Variables que vamos a usar:
- el promedio de energy, valence, danceability, acousticness, instrumentalness y tempo de lo que escucha;
- el desvío de energy y valence, para ver si escucha siempre lo mismo o cosas variadas;
- cuántos artistas distintos escucha;
- qué parte de sus escuchas son de su artista favorito;
- la popularidad media de los artistas que escucha. Usamos el logaritmo de los usuarios para que los
  artistas que tienen mas popularidad no pesen tanto.

In [4]:
consulta_usuarios = """
-- Escuchas de canciones con datos de audio
WITH escuchas AS (
    SELECT e.user_id, e.artist_id,
           a.energy, a.valence, a.danceability, a.acousticness,
           a.instrumentalness, a.tempo,
           LOG10(o.oyentes) AS popularidad
    FROM events_clean e
    JOIN acoustic_clean  a ON e.track_id  = a.track_id
    JOIN oyentes_artista o ON e.artist_id = o.artist_id
),

-- Buscamos la cantidad de escuchas al artista más escuchado
por_artista AS (
    SELECT user_id, artist_id, COUNT(*) AS n
    FROM escuchas
    GROUP BY user_id, artist_id
),
favorito AS (
    SELECT user_id, MAX(n) AS escuchas_top
    FROM por_artista
    GROUP BY user_id
),

-- Calculamos las variables por usuario
resumen AS (
    SELECT user_id,
           COUNT(*)                  AS total_escuchas,
           COUNT(DISTINCT artist_id) AS artistas_distintos,
           AVG(energy)               AS energy_media,
           STDDEV(energy)            AS energy_std,
           AVG(valence)              AS valence_media,
           STDDEV(valence)           AS valence_std,
           AVG(danceability)         AS dance_media,
           AVG(acousticness)         AS acoustic_media,
           AVG(instrumentalness)     AS instrumental_media,
           AVG(tempo)                AS tempo_medio,
           AVG(popularidad)          AS popularidad_media
    FROM escuchas
    GROUP BY user_id
    HAVING COUNT(*) >= 100
)

-- Agregamos los datos del usuario y la proporción del artista favorito
SELECT r.*,
       f.escuchas_top * 1.0 / r.total_escuchas AS pct_artista_top,
       u.country, u.age, u.gender
FROM resumen r
JOIN favorito f         ON r.user_id = f.user_id
LEFT JOIN users_clean u ON r.user_id = u.user_id
"""

con.execute(f"COPY ({consulta_usuarios}) TO {ruta_sql('vista_usuarios')} (FORMAT parquet)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Revisamos cuántos usuarios quedaron, si hay nulos y cómo son las variables. `total_escuchas` cuenta solo
las escuchas de canciones que tienen datos de audio.

In [5]:
vista_usuarios = q(f"SELECT * FROM {ruta_sql('vista_usuarios')}")

print("Usuarios en la vista:", len(vista_usuarios))
print()
print(vista_usuarios.isna().sum())

vista_usuarios.describe().T.round(3)

Usuarios en la vista: 53289

user_id                   0
total_escuchas            0
artistas_distintos        0
energy_media              0
energy_std                0
valence_media             0
valence_std               0
dance_media               0
acoustic_media            0
instrumental_media        0
tempo_medio               0
popularidad_media         0
pct_artista_top           0
country                   0
age                   11961
gender                    0
dtype: int64


,count,mean,std,min,25%,50%,75%,max
user_id,53289.0,23144265.899623,16094451.221663,384.0,8065274.0,19857764.0,38124557.0,50900118.0
total_escuchas,53289.0,6580.227345,9081.736118,100.0,1534.0,5310.0,8865.0,345014.0
artistas_distintos,53289.0,337.370058,366.793969,1.0,113.0,229.0,434.0,13354.0
energy_media,53289.0,0.692687,0.117028,0.102642,0.615357,0.694493,0.778673,0.964863
energy_std,53289.0,0.213505,0.039268,0.010329,0.188542,0.218539,0.241184,0.406878
valence_media,53289.0,0.424335,0.097475,0.043585,0.361969,0.429971,0.489383,0.844944
valence_std,53289.0,0.230226,0.027479,0.045913,0.215965,0.23517,0.24894,0.364258
dance_media,53289.0,0.495884,0.0791,0.103918,0.444998,0.499692,0.548037,0.861681
acoustic_media,53289.0,0.211274,0.132327,0.000147,0.108324,0.192218,0.288744,0.980609
instrumental_media,53289.0,0.214657,0.153411,0.0,0.093846,0.180581,0.30571,0.935433


In [10]:
vista_usuarios.head()

,user_id,total_escuchas,artistas_distintos,energy_media,energy_std,valence_media,valence_std,dance_media,acoustic_media,instrumental_media,tempo_medio,popularidad_media,pct_artista_top,country,age,gender
0,25613373,6165,320,0.826586,0.234239,0.235959,0.181831,0.375339,0.103164,0.421585,126.328817,2.866060,0.090998,FI,18,m
1,20611959,12330,1177,0.517438,0.237615,0.516609,0.242264,0.551298,0.450131,0.094705,119.826431,3.044116,0.055637,CA,27,m
2,7940900,9204,518,0.567383,0.269198,0.397379,0.254239,0.469914,0.344746,0.281546,121.437658,3.078096,0.048674,FI,<NA>,f
3,2730929,8696,1228,0.618630,0.270834,0.527572,0.257847,0.526635,0.333050,0.266064,122.204408,2.442079,0.057268,RU,35,m
4,1058825,5768,314,0.839592,0.179080,0.329070,0.189386,0.399088,0.047493,0.185614,125.475708,3.406815,0.031553,NL,31,m


**Observaciones:**

Quedaron 53.289 de los 55.190 usuarios (96,6%). La edad tiene 11.961 nulos, pero no la usamos para
entrenar.

Las escalas son muy distintas: el tempo anda por 124, los artistas distintos tienen una mediana de 229
y la mayoría de las demás variables va de 0 a 1. Por eso hay que estandarizar antes de K-means.

Los artistas distintos llegan a 13.354, muy lejos de la mediana, así que seguramente convenga usar el
logaritmo. El tempo medio casi no cambia entre usuarios, vamos a ver en el clustering si sirve o no.

Hay usuarios con `pct_artista_top` igual a 1, o sea que escuchan a un solo artista. Los dejamos.

## 4. Vista para recomendación: `vista_usuario_artista`

Cada fila es un usuario, un artista y cuántas veces lo escuchó. Si alguien escuchó 15 veces a Coldplay
y 4 a Queen, aparece en dos filas.

Usamos los mismos usuarios que en el clustering y solo los 10.000 artistas con más oyentes. Con todos
los artistas la matriz sería enorme, y a un artista que escucharon dos personas es muy difícil
recomendarlo.

Esta tabla resume todo el historial y no guarda las fechas.
La selección de usuarios y artistas para evaluar el recomendador también tiene que hacerse con los datos de entrenamiento.

In [7]:
consulta_ua = f"""
-- Selección de artistas
WITH top_artistas AS (
    SELECT artist_id
    FROM oyentes_artista
    ORDER BY oyentes DESC
    LIMIT 10000
),
-- Usuarios de la vista de clustering
usuarios_ok AS (
    SELECT user_id FROM {ruta_sql('vista_usuarios')}
)
-- Escuchas por usuario y artista
SELECT e.user_id, e.artist_id, COUNT(*) AS escuchas
FROM events_clean e
JOIN usuarios_ok  u ON e.user_id   = u.user_id
JOIN top_artistas t ON e.artist_id = t.artist_id
GROUP BY e.user_id, e.artist_id
"""

con.execute(f"COPY ({consulta_ua}) TO {ruta_sql('vista_usuario_artista')} (FORMAT parquet)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Calculamos la densidad: qué porcentaje de todas las combinaciones usuario-artista posibles tiene
alguna escucha. Si es baja, en el recomendador conviene usar una matriz dispersa (que guarda solo los
valores distintos de cero).

In [8]:
resumen_ua = q(f"""
    SELECT COUNT(*)                  AS pares,
           COUNT(DISTINCT user_id)   AS usuarios,
           COUNT(DISTINCT artist_id) AS artistas,
           MEDIAN(escuchas)          AS mediana_escuchas,
           MAX(escuchas)             AS max_escuchas
    FROM {ruta_sql('vista_usuario_artista')}
""")

resumen_ua["densidad_%"] = resumen_ua["pares"] / (resumen_ua["usuarios"] * resumen_ua["artistas"]) * 100
resumen_ua

,pares,usuarios,artistas,mediana_escuchas,max_escuchas,densidad_%
0,12691413,53271,10000,4.0,181440,2.382424


In [9]:
# Primeras filas, ordenadas para ver juntos los artistas de cada usuario.
# No la cargamos entera porque tiene más de 12 millones de filas.
q(f"""
    SELECT *
    FROM {ruta_sql('vista_usuario_artista')}
    ORDER BY user_id, escuchas DESC
    LIMIT 10
""")

,user_id,artist_id,escuchas
0,384,67,651
1,384,46553,477
2,384,17352,458
3,384,2583,384
4,384,15202,362
5,384,9037,348
6,384,7993,320
7,384,1996,299
8,384,47,294
9,384,362,241


**Observaciones:**

Quedaron 12.691.416 pares de 53.271 usuarios y 10.000 artistas. Hay 18 usuarios del clustering que no
aparecen porque no escuchan a ninguno de esos artistas.

La densidad es de 2,38%: de cada 100 combinaciones, solo 2 o 3 tienen escuchas. La mediana es 4
escuchas por par y el máximo 181.440, así que en el recomendador vamos a probar con `log1p(escuchas)`
(el logaritmo de 1 + escuchas) para que los valores altos no tapen al resto.

Las dos quedan en `data/vistas/` y son las que usamos en los notebooks de cada modelo.